In [ ]:
pip install geopandas pandas numpy requests shapely

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/TireTrace')

Mounted at /content/drive


# STEP 1: Load + clip roads

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import requests
import json
import os
from shapely.geometry import Point
from shapely.ops import nearest_points

ROADS_SHP     = "data/roads/EOTROADS_ARC.shp"
NHD_SHP       = "data/nhd/NHDFlowline.shp"
OUTPUT_PATH   = "data/output/road_runoff_scores.geojson"
NOAA_TOKEN    = "TXeGLpwnFQdfkwLZiCxoBpywwtejARtl"

# Bounding box for Merrimack River watershed
BBOX = {
    "min_lat": 42.55,
    "max_lat": 42.85,
    "min_lon": -71.45,
    "max_lon": -70.95,
}

def load_roads(path):
    print("Loading MassGIS roads...")
    roads = gpd.read_file(path)
    roads = roads.to_crs(epsg=4326)
    roads = roads.cx[BBOX["min_lon"]:BBOX["max_lon"],
                     BBOX["min_lat"]:BBOX["max_lat"]]

    required_cols = ["AADT", "geometry", "STREETNAME", "RDTYPE"]
    missing = [c for c in required_cols if c not in roads.columns]
    if missing:
        print(f"WARNING: These expected columns not found: {missing}")
        print(f"Your actual columns: {roads.columns.tolist()}")
        print("Check the Road Inventory Data Dictionary PDF for correct field names.")

    roads = roads[roads["AADT"].notna() & (roads["AADT"] > 0)]

    roads_proj = roads.to_crs(epsg=26986)
    roads["length_km"] = roads_proj.geometry.length / 1000

    print(f"  Loaded {len(roads)} road segments with AADT data")
    return roads

# STEP 2: Load waterways (NHD flowlines)

In [3]:
def load_waterways(path):
    print("Loading NHD waterways...")
    water = gpd.read_file(path)
    water = water.to_crs(epsg=4326)

    water = water.cx[BBOX["min_lon"]:BBOX["max_lon"],
                     BBOX["min_lat"]:BBOX["max_lat"]]

    # NHD FTYPE codes: 460=Stream, 558=Artificial Path, 336=Canal/Ditch
    if "FTYPE" in water.columns:
        water = water[water["FTYPE"].isin([460, 558, 336, 420])]

    print(f"  Loaded {len(water)} waterway segments")
    return water

# STEP 3: Calculate distance from each road centroid to nearest waterway

In [4]:
def calc_road_to_water_distance(roads, waterways):
    print("Calculating road → waterway distances (this takes a few minutes)...")

    roads_proj    = roads.to_crs(epsg=26986)
    water_proj    = waterways.to_crs(epsg=26986)

    water_union = water_proj.geometry.union_all()

    distances = []
    for idx, row in roads_proj.iterrows():
        road_centroid = row.geometry.centroid
        nearest_pt    = nearest_points(road_centroid, water_union)[1]
        dist_m        = road_centroid.distance(nearest_pt)
        distances.append(dist_m)

    roads["dist_to_water_m"] = distances
    print(f"  Done. Avg distance to waterway: {np.mean(distances):.0f}m")
    return roads


# STEP 4: Fetch average monthly rain days from NOAA for Boston Logan

In [5]:
def get_avg_rain_days():
    print("Fetching NOAA rainfall data...")

    if NOAA_TOKEN == "YOUR_NOAA_TOKEN_HERE":
        print("  No NOAA token set — using default 11 rain days/month for MA")
        return 11.0

    url = "https://www.ncdc.noaa.gov/cdo-web/api/v2/data"
    headers = {"token": NOAA_TOKEN}
    params = {
        "datasetid":  "GHCND",
        "stationid":  "GHCND:USW00014739",
        "datatypeid": "PRCP",
        "startdate":  "2023-01-01",
        "enddate":    "2023-12-31",
        "limit":      1000,
        "units":      "metric",
    }

    try:
        r = requests.get(url, headers=headers, params=params, timeout=10)
        data = r.json()
        daily = pd.DataFrame(data["results"])
        daily["date"] = pd.to_datetime(daily["date"])
        daily["month"] = daily["date"].dt.month
        rain_days = daily[daily["value"] > 0.0].groupby("month").size()
        avg = rain_days.mean()
        print(f"  Average rain days/month (Boston Logan 2023): {avg:.1f}")
        return avg
    except Exception as e:
        print(f"  NOAA fetch failed ({e}) — using default 11 rain days/month")
        return 11.0


# STEP 5: Physics-informed runoff model

In [6]:

# Based on:
#   Pierson et al. 2020 (Science of the Total Environment)
#   Tire wear emission factor: ~0.1 g per vehicle per km
#   6PPD-quinone release: 10% of total tire wear mass
#
# Runoff fraction uses exponential distance decay:
#   closer to waterway = more particles reach it
#   steeper slope = faster runoff (approximated from road type)

def calc_runoff_scores(roads, rain_days_per_month):
    print("Calculating tire particle runoff scores...")

    EMISSION_FACTOR_G_PER_VEH_KM = 0.1
    PPD_FRACTION = 0.10

    # Slope proxy from road type (RDTYPE field):
    # 1=Interstate, 2=US Route, 3=State Route, 4=State Numbered, 5=Local, 6=Private
    slope_by_rdtype = {1: 1.15, 2: 1.10, 3: 1.05, 4: 1.0, 5: 0.95, 6: 0.90}
    if "RDTYPE" in roads.columns:
        roads["slope_factor"] = roads["RDTYPE"].map(slope_by_rdtype).fillna(1.0)
    else:
        roads["slope_factor"] = 1.0

    roads["runoff_fraction"] = np.exp(-roads["dist_to_water_m"] / 200.0)

    # Total tire particles emitted per segment per day (grams)
    roads["emission_g_per_day"] = (
        roads["AADT"] *
        roads["length_km"] *
        EMISSION_FACTOR_G_PER_VEH_KM
    )

    # Particles reaching waterway per rain event (grams)
    roads["runoff_g_per_rain"] = (
        roads["emission_g_per_day"] *
        roads["runoff_fraction"] *
        roads["slope_factor"]
    )

    # Monthly load estimate (mg). convert g → mg (*1000)
    roads["runoff_mg_per_month"] = (
        roads["runoff_g_per_rain"] *
        rain_days_per_month *
        1000
    )

    # Toxic 6PPD-quinone fraction (mg/month)
    roads["ppd_mg_per_month"] = roads["runoff_mg_per_month"] * PPD_FRACTION

    # Normalize to 0–100 score for Flutter display
    max_val = roads["runoff_mg_per_month"].quantile(0.95)
    roads["pollution_score"] = (
        (roads["runoff_mg_per_month"] / max_val) * 100
    ).clip(0, 100).round(1)

    print(f"  Top 5 worst road segments:")
    top5 = roads.nlargest(5, "runoff_mg_per_month")[
        ["STREETNAME", "AADT", "dist_to_water_m", "runoff_mg_per_month", "ppd_mg_per_month"]
    ]
    print(top5.to_string(index=False))

    return roads

#STEP 6A

In [7]:
def add_city_names(roads):
    print("Assigning city names...")

    # Approximate bounding boxes for Merrimack watershed cities
    city_bounds = [
        ("Lawrence",      42.685, 42.730, -71.215, -71.145),
        ("Lowell",        42.615, 42.665, -71.380, -71.285),
        ("Haverhill",     42.750, 42.800, -71.130, -71.045),
        ("Andover",       42.640, 42.700, -71.175, -71.110),
        ("Methuen",       42.710, 42.760, -71.250, -71.165),
        ("North Andover", 42.680, 42.710, -71.140, -71.090),
        ("Dracut",        42.660, 42.700, -71.340, -71.270),
        ("Tewksbury",     42.595, 42.645, -71.260, -71.195),
        ("Chelmsford",    42.580, 42.640, -71.380, -71.310),
    ]

    roads_4326 = roads.to_crs(epsg=4326)
    cities = []

    for _, row in roads_4326.iterrows():
        centroid = row.geometry.centroid
        lat, lng = centroid.y, centroid.x
        city = "Merrimack Region"
        for name, min_lat, max_lat, min_lng, max_lng in city_bounds:
            if min_lat <= lat <= max_lat and min_lng <= lng <= max_lng:
                city = name
                break
        cities.append(city)

    roads["city"] = cities
    print(f"  City distribution:\n{pd.Series(cities).value_counts()}")
    return roads

def add_waterway_names(roads, waterways):
    print("Adding nearest waterway names...")

    name_col = next(
        (c for c in ["GNIS_NAME", "GNIS_Nam", "FULLNAME", "NAME"] if c in waterways.columns),
        None
    )

    if name_col is None:
        print("  No name column found in waterways — defaulting to 'Merrimack River'")
        roads["nearest_waterway"] = "Merrimack River"
        return roads

    roads_proj  = roads.to_crs(epsg=26986)
    water_proj  = waterways.to_crs(epsg=26986)
    water_proj  = water_proj[water_proj[name_col].notna() & (water_proj[name_col].str.strip() != "")]

    waterway_names = []
    for _, road_row in roads_proj.iterrows():
        centroid   = road_row.geometry.centroid
        distances  = water_proj.geometry.distance(centroid)
        nearest    = water_proj.loc[distances.idxmin(), name_col]
        waterway_names.append(nearest if nearest else "Merrimack River")

    roads["nearest_waterway"] = waterway_names
    print(f"  Unique waterways found: {set(waterway_names)}")
    return roads

# STEP 6: Export to GeoJSON for Flutter

In [8]:
def export_geojson(roads, output_path):
    print(f"Exporting to {output_path}...")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    export_cols = [
        "geometry",
        "STREETNAME",
        "AADT",
        "dist_to_water_m",
        "runoff_mg_per_month",
        "ppd_mg_per_month",
        "pollution_score",
        "length_km",
        "city",
        "nearest_waterway"
    ]

    export_cols = [c for c in export_cols if c in roads.columns]
    out = roads[export_cols].copy()

    for col in ["dist_to_water_m", "runoff_mg_per_month", "ppd_mg_per_month", "length_km"]:
        if col in out.columns:
            out[col] = out[col].round(2)

    out.to_file(output_path, driver="GeoJSON")

    size_kb = os.path.getsize(output_path) / 1024
    print(f"  Saved. File size: {size_kb:.0f} KB")
    print(f"  Total segments exported: {len(out)}")
    print(f"\nDone! Load {output_path} in Flutter with flutter_map's GeoJSON layer.")



# STEP 7: Quick sanity check + spot test

In [9]:
def spot_check(roads, street_name):
    """Check results for a specific road name.
    Shows top 3 matches sorted by AADT (busiest first)
    so you always get the main road, not a side street."""

    # Exact word match
    exact = roads[roads["STREETNAME"].str.fullmatch(street_name, case=False, na=False)]
    matches = exact if not exact.empty else roads[
        roads["STREETNAME"].str.contains(street_name, case=False, na=False)
    ]

    if matches.empty:
        print(f"No roads found matching '{street_name}'")
        return

    # Sort by AADT so we always show the busiest/most important segment first
    matches = matches.sort_values("AADT", ascending=False)

    print(f"\n--- Spot check: '{street_name}' --- showing top 3 segments by traffic ---")
    for _, row in matches.head(3).iterrows():
        print(f"\n  Road name:                  {row.get('STREETNAME', 'Unknown')}")
        print(f"  AADT (daily cars):          {row['AADT']:,.0f}")
        print(f"  Length:                     {row['length_km']:.2f} km")
        print(f"  Distance to waterway:       {row['dist_to_water_m']:.0f} m")
        print(f"  Runoff fraction:            {row['runoff_fraction']:.3f} ({row['runoff_fraction']*100:.1f}% reaches water)")
        print(f"  Tire particles/month:       {row['runoff_mg_per_month']:,.0f} mg")
        print(f"  of which 6PPD-quinone:      {row['ppd_mg_per_month']:,.0f} mg")
        print(f"  Pollution score (0-100):    {row['pollution_score']}")
        print(f"  {'─'*45}")

# MAIN

In [10]:
if __name__ == "__main__":
    for path, name in [(ROADS_SHP, "Roads shapefile"), (NHD_SHP, "NHD waterways")]:
        if not os.path.exists(path):
            print(f"ERROR: {name} not found at '{path}'")
            print("Download instructions at the top of this file.")
            exit(1)

    roads = load_roads(ROADS_SHP)
    waterways = load_waterways(NHD_SHP)
    roads = calc_road_to_water_distance(roads, waterways)
    rain_days = get_avg_rain_days()
    roads = calc_runoff_scores(roads, rain_days)
    roads = add_city_names(roads)
    roads = add_waterway_names(roads, waterways)

    spot_check(roads, "River Road")
    spot_check(roads, "Merrimack")

    export_geojson(roads, OUTPUT_PATH)

Loading MassGIS roads...
  Loaded 24806 road segments with AADT data
Loading NHD waterways...


/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D LineString' is converted to 'LineString Z'
  return ogr_read(


  Loaded 4558 waterway segments
Calculating road → waterway distances (this takes a few minutes)...
  Done. Avg distance to waterway: 1408m
Fetching NOAA rainfall data...
  Average rain days/month (Boston Logan 2023): 11.0
Calculating tire particle runoff scores...
  Top 5 worst road segments:
                STREETNAME   AADT  dist_to_water_m  runoff_mg_per_month  ppd_mg_per_month
BLUE STAR MEMORIAL HIGHWAY 120002        39.411433         3.639164e+08      3.639164e+07
BLUE STAR MEMORIAL HIGHWAY 118023        59.509367         3.589324e+08      3.589324e+07
BLUE STAR MEMORIAL HIGHWAY 134147        75.492176         2.799973e+08      2.799973e+07
BLUE STAR MEMORIAL HIGHWAY  78205        38.148749         2.565201e+08      2.565201e+07
                   ROUTE 3  94403        35.671308         2.395987e+08      2.395987e+07
Assigning city names...
  City distribution:
Merrimack Region    13336
Lowell               3334
Lawrence             2242
Haverhill            1909
Andover         